# grads-dict-accumulate-parents — faded example 1: Faded: use .get(parent, 0) in accumulate_into_grads

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `grads-dict-accumulate-parents`. The last cell reports your progress on the `Backprop: grads dict accumulate parents` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: grads dict accumulate parents` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`grads-dict-accumulate-parents`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "grads-dict-accumulate-parents"
DD_SUBTOPIC = "Backprop: grads dict accumulate parents"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The accumulation line in the reverse pass is `grads[parent] = grads.get(parent, 0) + g`. The `.get(parent, 0)` idiom is critical: it returns 0 (the additive identity) when `parent` is not yet in the dict, avoiding a `KeyError` on the first visit. Using `grads[parent] + g` directly would raise `KeyError`. Using `grads[parent] = g` (overwrite) would discard previous contributions from other paths.

## Faded exercise 1

Complete `accumulate_into_grads` by filling in the right-hand side of the assignment inside the loop. The blank is the expression that correctly handles both first-visit and revisit cases.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t

t.manual_seed(0)

class Node:
    def __init__(self, name):
        self.name = name

def accumulate_into_grads(grads, contributions):
    for parent, g in contributions:
        grads[parent] = None  # TODO: fill in this step — read the prompt cell above

# Exercise it
p = Node('x')
grads = {}
accumulate_into_grads(grads, [(p, t.tensor([1.0, 2.0]))])
accumulate_into_grads(grads, [(p, t.tensor([3.0, 4.0]))])
print(grads[p])  # should be [4.0, 6.0]


def _test():
    import torch as t

    class Node:
        def __init__(self, n):
            self.name = n

    p1 = Node('p1')
    p2 = Node('p2')

    grads = {}
    # First visit to p1 (dict is empty -> must not KeyError)
    accumulate_into_grads(grads, [(p1, t.tensor([1.0, 0.0]))])
    assert t.allclose(grads[p1], t.tensor([1.0, 0.0])), "First visit failed"

    # Second visit to p1 (should accumulate)
    accumulate_into_grads(grads, [(p1, t.tensor([2.0, 3.0]))])
    assert t.allclose(grads[p1], t.tensor([3.0, 3.0])), "Second visit failed"

    # First visit to p2 via same call as third visit to p1
    accumulate_into_grads(grads, [(p1, t.tensor([0.5, 0.5])), (p2, t.tensor([7.0, 7.0]))])
    assert t.allclose(grads[p1], t.tensor([3.5, 3.5]))
    assert t.allclose(grads[p2], t.tensor([7.0, 7.0]))


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

t.manual_seed(0)

class Node:
    def __init__(self, name):
        self.name = name

def accumulate_into_grads(grads, contributions):
    for parent, g in contributions:
        grads[parent] = grads.get(parent, 0) + g

# Exercise it
p = Node('x')
grads = {}
accumulate_into_grads(grads, [(p, t.tensor([1.0, 2.0]))])
accumulate_into_grads(grads, [(p, t.tensor([3.0, 4.0]))])
print(grads[p])  # should be [4.0, 6.0]
```
</details>